# Capturas MQTT - hardware eficiente

Notebook para tratar os dados captados pelo firmware eficiente e gravados por `mqtt_csv_logger.py` em `mqtt_dados_eficiente.csv`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

CSV_PATH = Path("mqtt_dados_eficiente.csv")
TRATADO_PATH = Path("mqtt_dados_eficiente_tratado.csv")

COLUNAS = [
    "received_at",
    "topic",
    "device_id",
    "algorithm",
    "sequence",
    "payload_bytes",
    "packet_index",
    "source_mac",
    "rssi",
    "channel",
    "frequency",
    "frame_type",
    "seen_count",
    "parse_error",
    "raw_payload",
]

def carregar_csv(path: Path = CSV_PATH) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame(columns=COLUNAS)
    return pd.read_csv(path)

df = carregar_csv()
df.head()

## Limpeza e normalização

In [ ]:
def normalizar_capturas(df: pd.DataFrame) -> pd.DataFrame:
    tratado = df.copy()
    for coluna in COLUNAS:
        if coluna not in tratado.columns:
            tratado[coluna] = pd.NA

    tratado = tratado[COLUNAS]
    tratado["received_at"] = pd.to_datetime(tratado["received_at"], errors="coerce", utc=True)

    numericas = ["sequence", "payload_bytes", "packet_index", "rssi", "channel", "frequency", "seen_count"]
    for coluna in numericas:
        tratado[coluna] = pd.to_numeric(tratado[coluna], errors="coerce")

    texto = ["topic", "device_id", "algorithm", "source_mac", "frame_type", "parse_error", "raw_payload"]
    for coluna in texto:
        tratado[coluna] = tratado[coluna].fillna("").astype(str)

    tratado["has_parse_error"] = tratado["parse_error"].str.len() > 0
    tratado["is_valid_packet"] = (~tratado["has_parse_error"]) & tratado["source_mac"].str.len().gt(0)
    return tratado

tratado = normalizar_capturas(df)
validos = tratado[tratado["is_valid_packet"]].copy()
tratado.to_csv(TRATADO_PATH, index=False)
print(f"Linhas totais: {len(tratado)}")
print(f"Pacotes validos: {len(validos)}")
print(f"CSV tratado salvo em: {TRATADO_PATH.resolve()}")
tratado.head()

## Resumo das capturas

In [ ]:
resumo = pd.DataFrame([
    {
        "mensagens_mqtt": tratado["sequence"].dropna().nunique(),
        "linhas_csv": len(tratado),
        "pacotes_validos": len(validos),
        "macs_unicos": validos["source_mac"].nunique(),
        "rssi_medio": validos["rssi"].mean(),
        "rssi_min": validos["rssi"].min(),
        "rssi_max": validos["rssi"].max(),
        "erros_parse": int(tratado["has_parse_error"].sum()),
    }
])
resumo

In [ ]:
canais = (
    validos.groupby("channel", dropna=True)
    .agg(pacotes=("source_mac", "count"), macs_unicos=("source_mac", "nunique"), rssi_medio=("rssi", "mean"))
    .sort_values("pacotes", ascending=False)
)
canais

## Gráficos

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print("matplotlib nao esta instalado; instale com `pip install matplotlib` para gerar os graficos.")

if validos.empty:
    print("Sem pacotes validos para plotar.")
elif plt is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    validos["rssi"].dropna().plot(kind="hist", bins=12, ax=axes[0], color="#2F6CB3", edgecolor="white")
    axes[0].set_title("Distribuicao de RSSI")
    axes[0].set_xlabel("RSSI (dBm)")

    validos["channel"].dropna().value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#D69A2D")
    axes[1].set_title("Pacotes por canal")
    axes[1].set_xlabel("Canal")
    axes[1].set_ylabel("Pacotes")

    top_macs = validos.groupby("source_mac")["seen_count"].sum().sort_values(ascending=False).head(10)
    top_macs.sort_values().plot(kind="barh", ax=axes[2], color="#C24B3A")
    axes[2].set_title("Top MACs por seen_count")
    axes[2].set_xlabel("seen_count acumulado")

    fig.tight_layout()
    plt.show()